# Environment

In [1]:
import os
os.chdir("..")

In [2]:
import pandas as pd
import numpy as np
from collections import deque
import re

from src.utils.path import DATA_PATH

# Load Data

In [12]:
mol = pd.read_csv(f"{DATA_PATH}/lexicons/mol.csv")

# Feature Extraction Functions

## Algorithm Corasick

In [6]:
class AhoNode:
    def __init__(self):
        self.children = {}
        self.fail = None
        self.output = []

In [7]:
class AhoCorasick:
    def __init__(self):
        self.root = AhoNode()

    def add_expression(self, expression, metadata):
        clean_expr = re.sub(r'[^\w\s]', '', expression.lower())
        words = clean_expr.split()
        if not words: return

        node = self.root
        for word in words:
            if word not in node.children:
                node.children[word] = AhoNode()
            node = node.children[word]

        match_info = {
            "content": expression,
            **metadata
        }
        node.output.append(match_info)

    def build(self):
        queue = deque()
        for word, child in self.root.children.items():
            child.fail = self.root
            queue.append(child)

        while queue:
            u = queue.popleft()
            for word, v in u.children.items():
                f = u.fail
                while word not in f.children and f != self.root:
                    f = f.fail

                v.fail = f.children[word] if word in f.children else self.root
                v.output.extend(v.fail.output)
                queue.append(v)

    def search(self, text):
        if not isinstance(text, str): return []

        clean_text = re.sub(r'[^\w\s]', '', text.lower())
        words = clean_text.split()
        node = self.root
        results = []

        for i, word in enumerate(words):
            while word not in node.children and node != self.root:
                node = node.fail

            node = node.children.get(word, self.root)

            for found in node.output:
                item = found.copy()
                results.append(item)

        return results

In [8]:
aho = AhoCorasick()

In [13]:
for row in mol[["pt-brazilian-portuguese", "explicit-or-implicit", "term-or-expression", "pt-contextual-label", "pt-hate-label"]].values:
    expression = row[0]
    metadata = {
        "explicit_implicit": row[1],
        "term_type": row[2],
        "contextual_label": row[3],
        "hate_label": row[4]
    }
    aho.add_expression(expression, metadata)

In [14]:
aho.build()

## Compute MOL

$\text{Mol}_{xy} = \text{freq}_{xy} \cdot Wc_x \cdot Wh_x$


$
Wc_x =
\begin{cases}
1, & \text{se context-dependent} \\
2, & \text{caso contrário}
\end{cases}
$

$
Wh_x =
\begin{cases}
2, & \text{se possui hate label} \\
1, & \text{caso contrário}
\end{cases}
$

[Vargas et al. (2022)](https://drive.google.com/file/d/1p72roMNn9GGnDts-bljzSDRlCc7HhHpx/view?usp=sharing)

In [41]:
def extractFeature(df):
    scores = []
    
    for text, mol_terms in zip(df["text"], df["text"].apply(aho.search)):
        score = 0
        
        if mol_terms:
            word_count = len(text.split())
            if word_count == 0:
                scores.append(0.0)
                continue
                
            text_lower = text.lower()
            
            for t in mol_terms:
                freq = text_lower.count(t["content"]) / word_count
                wc = 1 if t["contextual_label"] else 2
                wh = 2 if t["hate_label"] else 1

                score += freq * wh * wc

        scores.append(round(score, 2))

    return pd.Series(scores, name="mol")

# Feature Engineering in Corpora

## Hate-BR

In [42]:
hate = pd.read_csv(f"{DATA_PATH}/hate-br/processed/hate-br.csv")

In [43]:
hate_mol = extractFeature(hate)

In [46]:
hate_mol.to_csv(f"{DATA_PATH}/hate-br/features/mol.csv", index=False)

## Ited-BR

In [29]:
ited = pd.read_csv(f"{DATA_PATH}/ited-br/processed/ited-br.csv")

In [ ]:
ited_mol = extractFeature(ited)

In [31]:
ited_mol.to_csv(f"{DATA_PATH}/ited-br/features/mol.csv", index=False)

## Unified-Hate

In [33]:
unified = pd.read_csv(f"{DATA_PATH}/unified-hate/processed/unified-hate.csv")

In [ ]:
unified_mol = extractFeature(unified)

In [37]:
unified_mol.to_csv(f"{DATA_PATH}/unified-hate/features/mol.csv", index=False)